# Vector fitting: from frequency data to circuit models

Vector fitting approximates sampled frequency responses with poles and residues that can be used in a circuit simulator. The [scikit-rf tutorial](https://scikit-rf.readthedocs.io/en/latest/tutorials/VectorFitting.html) explains the method and its equivalent circuits.

We'll use scikit-rf to fit S-parameter data, then load the coefficients into Circulax. The first three examples cover a passive two-port, a resonant four-port, and an active transmitter. Along the way, we'll check fitting error, pole stability, and passivity where it applies. Scikit-rf also has a [passivity example](https://scikit-rf.readthedocs.io/en/latest/examples/vectorfitting/vectorfitting_ex4_passivity.html).

### Adding propagation delays

A pure delay, $e^{-sd}$, can take many rational poles to approximate over a wide band. Circulax aims to reduce that pole count by removing known propagation delays before fitting and restoring them with transmission-line elements afterward. The [last example](#delay-aware) tests this with the same fitter and accuracy target on both sides of the comparison.

Currently, Circulax uses one delay per port around a rational core. The lines work in DC, AC, harmonic balance, and transient. This helps for the known-delay example; we haven't established a general advantage on measured networks. Simulation cost also depends on the extra line variables and stored delay history.

We intend to explore a more general method like [Chinea, Triverio, and Grivet-Talocia's Delayed Vector Fitting (2010)](https://www.waves.utoronto.ca/triverio/papers/jnl-2010-tadvp-vfdelays.pdf), which uses several delayed contributions to represent multiple paths and internal reflections. That extension is not implemented yet.

Run the notebook from top to bottom. Sections reuse names such as `network` and `ss`, so rerun a section from its beginning when returning to it. All data is local or bundled with scikit-rf. The four-port fits take longer than the other examples.

<a id="contents"></a>
## Contents

- [How fitted coefficients enter the circuit](#engineering)
  - [Accuracy, stability, and passivity](#admission)
  - [Delay de-embedding](#delay-fit-plan)
- [1. Ring-slot two-port](#ring-slot)
  - [Compare fitting orders](#ring-fit)
  - [Optional passivity correction](#ring-enforce)
  - [Compare errors and poles](#ring-check)
  - [Create a component](#ring-component)
  - [Save and reload](#ring-save)
- [2. Resonant four-port](#resonant)
  - [Compare errors](#resonant-accuracy)
  - [Apply passivity correction](#resonant-correction)
- [3. Active transmitter](#active)
  - [Fit the active device](#active-fit)
  - [Validate the fit](#active-validation)
- [4. Fit with explicit delays](#delay-aware)
- [Summary and references](#summary)


<a id="engineering"></a>
## How fitted coefficients enter the circuit

The proper rational S model used here is

$$
S(s)=D+\sum_k\frac{R_k}{s-p_k},\qquad s=j2\pi f.
$$

Circulax converts it to admittance,

$$
Y(s)=\frac{1}{z_0}(I-S(s))(I+S(s))^{-1},
$$

and adds states to the circuit equations:

$$
\dot{x}=A_Yx+B_Yv,\qquad i=C_Yx+D_Yv.
$$

During a transient, the solver integrates these states and includes their port currents in the current-balance equations. The fitted poles therefore affect the circuit's time response.

<a id="admission"></a>
### Accuracy, stability, and passivity

A close match on an S-parameter plot is only one check. Conversion to Y can introduce unstable poles through the inverse of $I+S$, even if the S poles are stable.

| Check | Why it matters |
| --- | --- |
| Error on held-out frequencies | Tests interpolation and reveals missed resonances |
| Stability of the converted Y poles | An excited mode with a positive pole real part grows in time |
| Passivity, for a passive device | A nonpassive fit can supply energy and disturb the loaded circuit |
| Behavior outside the measured band | DC and fast edges may reach frequencies the measurements don't cover |

For a pole $p=\alpha+j\beta$, the mode's amplitude varies as $e^{\alpha t}$. If $\alpha=10^9\,\mathrm{s}^{-1}$, an excited mode grows by about 22,000 times in 10 ns. Reducing the time step won't remove that growth. A solver can also struggle with a stable model, so convergence failure alone doesn't identify the cause.

For power-normalized S-parameters, passivity requires $\sigma_{\max}(S)\le1$ at every frequency, along with stable, causal behavior. The optional correction used here adjusts residues and the constant term while keeping the S poles fixed. We check error again afterward and use a rational passivity test to look for violations between sampled frequencies. A model that already passes needs no correction.

These checks don't supply missing measurements. For example, a fit to the ring-slot data at 75–110 GHz needs additional low-frequency information before we can trust a startup waveform. The examples below report model properties; they don't demonstrate the possible transient failures listed here. Loaded-circuit stability also depends on the source, load, and feedback. See the [engineering note](../../docs/rational_model_enforcement.md) for more detail.

<a id="delay-fit-plan"></a>
### Delay de-embedding

With a known one-way delay $d_i$ at each port, we use

$$
S_{\mathrm{full}}(s)=P(s)S_{\mathrm{core}}(s)P(s),
\qquad P(s)=\operatorname{diag}(e^{-s d_i}).
$$

`fit_model` removes those delays and fits the core. `component_from_coefficients` then connects the core to bidirectional `TransmissionLine` elements. Their shared `signals.at_delay(...)` equations work in all four solvers; transient accuracy depends on time stepping and history interpolation. The older `rational_delay_component` is a frequency-domain reference implementation. The experimental `fit_with_delay` examples below also retain their older delay convention.

Conventional vector fitting is the default core fitter. AAA initialization is a separate option. Automatic delay inference is optional too: phase slope can come from resonances, and transmission alone identifies a sum of port delays, not their allocation. Candidates are compared with an undelayed baseline and checked for accuracy, realization validity, and pole count.

A possible future extension is the more general form used by [Chinea et al.](https://www.waves.utoronto.ca/triverio/papers/jnl-2010-tadvp-vfdelays.pdf):

$$
H(s) \approx \sum_m Q_m(s)e^{-s\tau_m}.
$$

Several delays can represent separate arrivals and internal reflections. Implementing this would need delay identification and validation, followed by accuracy and simulation-cost comparisons with the current approach. The [implementation plan](../../docs/delay_aware_fitting_plan.md) records the current design and solver tests.

[Back to contents](#contents)


<a id="ring-slot"></a>
## 1. Ring-slot two-port

Start with the passive reciprocal two-port from scikit-rf's [ring-slot example](https://scikit-rf.readthedocs.io/en/latest/examples/vectorfitting/vectorfitting_ex1_ringslot.html). We'll reserve every fifth frequency for testing, compare fitting orders, and turn a suitable fit into a Circulax component.

[Back to contents](#contents)


In [ ]:
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import skrf
from skrf.vectorFitting import VectorFitting
from circulax.fitting import (
    ModelFitOptions, ModelCoefficients, fit_model, component_from_coefficients,
    scattering_state_space_to_admittance, surface_from_fit, validate_surface_fit,
)
from circulax.fitting.types import VFModel, vfmodel_to_ss

network = skrf.data.ring_slot
freqs, S = network.f, network.s
z0 = float(network.z0[0, 0].real)
train = np.arange(len(freqs)) % 5 != 0
holdout = ~train
print(f"{train.sum()} training / {holdout.sum()} held-out samples; {freqs[0]/1e9:g}–{freqs[-1]/1e9:g} GHz")


<a id="ring-fit"></a>
### 1. Compare fitting orders

Fit the same training samples with several prescribed orders and scikit-rf's automatic order selection. `fit_model` uses that same automatic fitter by default and returns the coefficients.

The reserved samples are used only for evaluation. The timers below measure individual fitting runs; they exclude conversion and validation and haven't been warmed up.


In [ ]:
baselines = {}
for label, order in [("scikit-rf fixed 3", 3), ("scikit-rf fixed 4", 4), ("scikit-rf auto", None)]:
    fitter = VectorFitting(network[train])
    started = time.perf_counter()
    if order is None:
        fitter.auto_fit()
    else:
        fitter.vector_fit(n_poles_real=order, n_poles_cmplx=0)
    elapsed = time.perf_counter() - started
    poles, residues = [], []
    for index, pole in enumerate(fitter.poles):
        residue = fitter.residues[:, index].reshape(2, 2)
        poles.append(pole)
        residues.append(residue)
        if pole.imag != 0:
            poles.append(pole.conjugate())
            residues.append(residue.conjugate())
    coefficients = ModelCoefficients(
        np.asarray(poles), np.stack(residues, axis=-1),
        fitter.constant_coeff.reshape(2, 2), z0,
    )
    baselines[label] = coefficients
    print(label, "poles:", len(poles), "fit ms:", elapsed * 1000)

started = time.perf_counter()
raw = fit_model(S[train], freqs[train], z0=z0, options=ModelFitOptions(
    normalized_rmse=8e-7, max_absolute_error=1e-5,
))
print("fit_model (scikit-rf auto):", len(raw.poles), "poles; fit ms:", (time.perf_counter() - started) * 1000)


<a id="ring-enforce"></a>
### 2. Optional passivity correction

The four-pole fit already passes rational passivity and Y-stability checks. We'll use it to build the component. Enabling `enforce_passivity=True` also exercises the correction routine, though this fit should need little or no adjustment.

We require a training NRMSE of at most 100 ppm and a maximum absolute S error of 0.001, including after correction. The correction routine checks a finite grid; the rational passivity test below checks for violations between those samples.

Correction doesn't always converge. In a separate run, the experimental optimizer reached its 300-iteration limit on scikit-rf's automatic seven-pole fit. Scikit-rf provides its own enforcement method, which we haven't compared here.


In [ ]:
started = time.perf_counter()
corrected = fit_model(
    S[train], freqs[train], z0=z0,
    options=ModelFitOptions(
        vector_fit_order=(4, 0),
        normalized_rmse=1e-4, max_absolute_error=1e-3,
        enforce_passivity=True,
    ),
)
print("Four-pole fit + optional enforcement ms:", (time.perf_counter() - started) * 1000)
print(corrected.metadata["enforcement"])
assert len(corrected.poles) == 4


<a id="ring-check"></a>
### 3. Compare errors and poles

The table reports held-out error, scikit-rf's rational passivity result, and the largest real part of a converted Y pole. A negative value in the last column means all Y modes decay.

The lowest-error fit doesn't necessarily give the best circuit model. In this example, changing the order also changes whether the model passes the physical checks. Those differences are hard to see in the in-band plots alone.


In [ ]:
def rational_test(coefficients):
    oracle = VectorFitting(network[train])
    indices = np.flatnonzero(coefficients.poles.imag >= 0)
    oracle.poles = coefficients.poles[indices]
    oracle.residues = coefficients.residues[:, :, indices].reshape(4, -1)
    oracle.constant_coeff = coefficients.D.real.ravel()
    oracle.proportional_coeff = np.zeros(4)
    return oracle.passivity_test()

def admittance(coefficients):
    model = VFModel(coefficients.poles, coefficients.residues, coefficients.D.real, np.zeros((2, 2)))
    ss, _ = scattering_state_space_to_admittance(vfmodel_to_ss(model, 2), z0=z0)
    return ss

np.testing.assert_allclose(raw.evaluate(freqs), baselines["scikit-rf auto"].evaluate(freqs), atol=1e-12)
models = {**baselines, "Circulax candidate 4": corrected}
qualification = {}
for label, coefficients in models.items():
    bands = rational_test(coefficients)
    ss = admittance(coefficients)
    error = np.linalg.norm(coefficients.evaluate(freqs[holdout]) - S[holdout]) / np.linalg.norm(S[holdout])
    max_y_pole = float(np.asarray(ss.A).real.max())
    qualification[label] = (len(bands) == 0, max_y_pole < 0)
    print(f"{label:24s} order={len(coefficients.poles)} holdout={error:.3e} "
          f"passive={not len(bands)} max Re(Y pole)={max_y_pole:.3e} rad/s")
    if len(bands):
        print("  passivity violation intervals (Hz):", bands)
assert qualification["scikit-rf fixed 4"] == (True, True)
assert qualification["Circulax candidate 4"] == (True, True)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
predicted = corrected.evaluate(freqs)
for row in range(2):
    for col in range(2):
        ax = axes[row, col]
        ax.plot(freqs / 1e9, 20 * np.log10(np.maximum(abs(S[:, row, col]), 1e-15)), label="measured")
        ax.plot(freqs / 1e9, 20 * np.log10(np.maximum(abs(predicted[:, row, col]), 1e-15)), "--", label="qualified four-pole model")
        ax.set_title(f"S{row + 1}{col + 1}")
        ax.set_xlabel("GHz")
        ax.set_ylabel("dB")
        ax.grid(True)
axes[0, 0].legend()
plt.tight_layout()


<a id="ring-component"></a>
### 4. Create a component

Check the selected fit's held-out error, reciprocity, sampled admittance passivity, asymptotic terms, and Y stability before constructing the component. These complement the rational passivity test above.

The three-pole fit has a converted Y pole with real part about +6.74×10⁸ rad/s. If excited, that mode grows by a factor of $e$ roughly every 1.48 ns. The four-pole fit has only decaying Y modes. These are conclusions from the poles; we haven't simulated a ring-slot transient here.

`component_from_coefficients` rejects unstable S or Y poles but doesn't certify global passivity. The cell below validates and constructs the component. Using it for startup or pulse simulations would also require suitable low/high-frequency data and checks with the intended source, load, and time steps.


In [ ]:
ss = admittance(corrected)
features = np.ones((1, 1))
surface = surface_from_fit(ss, np.zeros(2), 2 * np.pi * freqs[-1], z0=z0)
report = validate_surface_fit(
    surface, S[train][None], features, freqs[train],
    validation_S=S[holdout][None], validation_features=features, validation_freqs=freqs[holdout],
    passivity_features=features, passivity_freqs=np.linspace(0, freqs[-1], 801),
    simulation_frequency_range=(0, freqs[-1]),
)
print(report.summary())
report.raise_for_simulation()
RingSlot = component_from_coefficients(corrected, name="RingSlot")
ring_slot_component = RingSlot()
print("Circulax component ports:", RingSlot.ports)


<a id="ring-save"></a>
### 5. Save and reload

The archive stores S coefficients, reference impedance, fit settings, and diagnostics. Another process can load it and build the component without fitting again. Here we use a temporary file.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    archive = Path(directory) / "ring_slot.npz"
    corrected.save(archive)
    ReloadedRingSlot = component_from_coefficients(archive, name="ReloadedRingSlot")
    restored = ModelCoefficients.load(archive)
    np.testing.assert_allclose(restored.evaluate(freqs), corrected.evaluate(freqs))
print("Coefficient save/load and component generation: PASS")


### Ring-slot results

The four-pole scikit-rf fit passes the passivity and Y-stability checks, with about 35 ppm held-out normalized error. The automatic fit is more accurate in frequency but needs further work before its Y realization can be used.

AAA (adaptive Antoulas–Anderson) is also available through `ModelFitOptions(method="aaa")`. It builds a rational approximation whose poles can initialize vector fitting. An earlier AAA-initialized seven-pole correction reached about 2.47 ppm held-out error; that separate experiment is described in the [benchmark notes](../../benchmarks/fitting/README.md). These examples haven't shown a speed or accuracy advantage for AAA over scikit-rf.

To try a different order, rerun both the error and physical checks. The next examples use a [resonant four-port](#resonant) and an [active transmitter](#active).

[Back to contents](#contents)


<a id="resonant"></a>
## 2. Resonant four-port

`Agilent_E5071B.s4p` contains the narrow resonances used in scikit-rf's [spiky-response example](https://scikit-rf.readthedocs.io/en/latest/examples/vectorfitting/vectorfitting_ex3_Agilent_E5071B.html).

We'll compare its prescribed scikit-rf fit with Circulax's experimental AAA/Y fitting path, again reserving every fifth sample. Then we'll apply passivity correction and measure how much it changes the error. This section uses the older `fit_with_delay` interface; the coefficient API also supports explicit delays in S fitting.

[Back to contents](#contents)


In [ ]:
from pathlib import Path
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import skrf
from skrf.vectorFitting import VectorFitting

from circulax.fitting import (
    FitValidationError,
    evaluate_sparameter_model,
    evaluate_surface,
    fit_with_delay,
    project_surface_passive,
    surface_from_fit,
    validate_surface_fit,
)

plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})

### Load the data

The Touchstone file and its upstream BSD-3-Clause license are included in the repository. No download is needed. Its reference impedance is 75 Ω.


In [ ]:
candidates = [
    Path("examples/fitting/data/Agilent_E5071B.s4p"),
    Path("data/Agilent_E5071B.s4p"),
]
data_path = next(path for path in candidates if path.exists())
network = skrf.Network(data_path)
z0 = float(np.real(network.z0[0, 0]))

sample_index = np.arange(len(network.f))
validation_mask = sample_index % 5 == 0
training_mask = ~validation_mask

frequencies_train = network.f[training_mask]
S_train = network.s[training_mask]
frequencies_validation = network.f[validation_mask]
S_validation = network.s[validation_mask]

print(network)
print(f"Training samples: {training_mask.sum()}, held-out samples: {validation_mask.sum()}, z0: {z0:g} ohm")

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(12, 9), sharex=True)
for row in range(4):
    for col in range(4):
        magnitude_db = 20 * np.log10(np.maximum(np.abs(network.s[:, row, col]), 1e-8))
        axes[row, col].plot(network.f / 1e9, magnitude_db, lw=1)
        axes[row, col].set_title(f"S{row + 1}{col + 1}")
        if row == 3:
            axes[row, col].set_xlabel("Frequency (GHz)")
        if col == 0:
            axes[row, col].set_ylabel("Magnitude (dB)")
fig.suptitle("Agilent E5071B measurement: all 16 responses")
fig.tight_layout()

### Fit with AAA, delay extraction, and Y conversion

`fit_with_delay` estimates port delays, de-embeds them, converts S to Y, discovers poles with AAA, and fits residues with a common pole set. It returns a state-space model.

We'll leave passivity enforcement off for the first comparison. The small `delay_scale` limits the amount removed from a response with strong resonances; it doesn't establish that the estimated phase slope is a physical propagation delay.


In [ ]:
start = time.perf_counter()
ss_circulax, delay, fit_metadata = fit_with_delay(
    S_train,
    frequencies_train,
    z0=z0,
    tol=1e-6,
    delay_scale=0.1,
    enforce_passive=False,
    verbose=False,
)
circulax_seconds = time.perf_counter() - start

S_circulax_train = evaluate_sparameter_model(ss_circulax, frequencies_train, delay, z0)
S_circulax_validation = evaluate_sparameter_model(ss_circulax, frequencies_validation, delay, z0)
S_circulax_all = evaluate_sparameter_model(ss_circulax, network.f, delay, z0)

print(f"Pole count: {fit_metadata['pole_count']}")
print(f"Extracted delays (ps): {np.round(delay * 1e12, 3)}")
print(f"Elapsed: {circulax_seconds:.3f} s")

### Fit with scikit-rf

The upstream example uses one real pole and 26 complex-conjugate pairs, for order 53. This order is supplied explicitly, while the AAA path selects poles automatically. Both fits use the same training samples.


In [ ]:
training_network = skrf.Network(
    frequency=skrf.Frequency.from_f(frequencies_train, unit="Hz"),
    s=S_train,
    z0=network.z0[training_mask],
    name="Agilent_E5071B_training",
)
vf_skrf = VectorFitting(training_network)
with warnings.catch_warnings(record=True) as skrf_warnings:
    warnings.simplefilter("always")
    start = time.perf_counter()
    vf_skrf.vector_fit(n_poles_real=1, n_poles_cmplx=26)
    skrf_seconds = time.perf_counter() - start

def evaluate_skrf(vf, frequencies, n_ports=4):
    result = np.empty((len(frequencies), n_ports, n_ports), dtype=complex)
    for row in range(n_ports):
        for col in range(n_ports):
            result[:, row, col] = vf.get_model_response(row, col, frequencies)
    return result

S_skrf_train = evaluate_skrf(vf_skrf, frequencies_train)
S_skrf_validation = evaluate_skrf(vf_skrf, frequencies_validation)
S_skrf_all = evaluate_skrf(vf_skrf, network.f)

print(f"Model order: {vf_skrf.get_model_order(vf_skrf.poles)}")
print(f"Passive over scikit-rf's assessment domain: {vf_skrf.is_passive()}")
print(f"Elapsed: {skrf_seconds:.3f} s")
for warning in skrf_warnings:
    print(f"scikit-rf warning: {warning.message}")

<a id="resonant-accuracy"></a>
### Compare errors

Complex RMSE includes both real and imaginary errors. Maximum absolute error helps expose missed narrow resonances that contribute little to the average. Since the orders differ, these results compare the two configurations rather than the solvers at equal order.


In [ ]:
def error_metrics(prediction, target):
    error = np.abs(prediction - target)
    return np.sqrt(np.mean(error**2)), np.max(error)

comparison = {
    "Circulax AAA": (
        fit_metadata["pole_count"],
        *error_metrics(S_circulax_train, S_train),
        *error_metrics(S_circulax_validation, S_validation),
    ),
    "scikit-rf VF": (
        vf_skrf.get_model_order(vf_skrf.poles),
        *error_metrics(S_skrf_train, S_train),
        *error_metrics(S_skrf_validation, S_validation),
    ),
}
print(f"{'method':<18} {'order':>7} {'train RMSE':>13} {'train max':>13} {'holdout RMSE':>14} {'holdout max':>13}")
for method, values in comparison.items():
    order, train_rmse, train_max, holdout_rmse, holdout_max = values
    print(f"{method:<18} {order:7d} {train_rmse:13.4e} {train_max:13.4e} {holdout_rmse:14.4e} {holdout_max:13.4e}")

In [ ]:
selected = [(0, 0), (1, 0), (2, 2), (3, 0)]
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, (row, col) in zip(axes.flat, selected, strict=True):
    ax.plot(network.f / 1e9, 20 * np.log10(np.maximum(np.abs(network.s[:, row, col]), 1e-8)), label="data")
    ax.plot(network.f / 1e9, 20 * np.log10(np.maximum(np.abs(S_circulax_all[:, row, col]), 1e-8)), label="Circulax")
    ax.plot(network.f / 1e9, 20 * np.log10(np.maximum(np.abs(S_skrf_all[:, row, col]), 1e-8)), "--", label="scikit-rf")
    ax.scatter(frequencies_validation / 1e9, 20 * np.log10(np.maximum(np.abs(S_validation[:, row, col]), 1e-8)), s=8, color="black", label="held out")
    ax.set_title(f"S{row + 1}{col + 1}")
    ax.set_ylabel("Magnitude (dB)")
    ax.set_xlabel("Frequency (GHz)")
axes[0, 0].legend(ncols=2)
fig.tight_layout()

### Validate the fitted model

Wrapping the state-space model as a single-corner rational surface lets us use the surface validation routines. Passivity is checked on a denser grid than the training and test sets. `raise_for_simulation()` rejects a result that fails the required checks.


In [ ]:
feature = np.ones((1, 1))
omega_scale = 2 * np.pi * network.f.max()
surface = surface_from_fit(ss_circulax, delay, omega_scale, z0=z0)
passivity_frequencies = np.linspace(network.f.min(), network.f.max(), 801)

report = validate_surface_fit(
    surface,
    S_train[None, ...],
    feature,
    frequencies_train,
    validation_S=S_validation[None, ...],
    validation_features=feature,
    validation_freqs=frequencies_validation,
    passivity_features=feature,
    passivity_freqs=passivity_frequencies,
    simulation_frequency_range=(network.f.min(), network.f.max()),
)
print(report.summary())
try:
    report.raise_for_simulation()
except FitValidationError as error:
    print(f"\nSimulation blocked as intended:\n{error}")

<a id="resonant-correction"></a>
### Apply passivity correction

The prototype projection shifts the diagonal of D and E just enough to meet the passivity constraint on the requested grid. It leaves the poles unchanged, so it can't recover dynamics that the fit missed. The second report measures the error after those shifts.


In [ ]:
passive_surface, shifts = project_surface_passive(surface, feature, passivity_frequencies)
passive_report = validate_surface_fit(
    passive_surface,
    S_train[None, ...],
    feature,
    frequencies_train,
    validation_S=S_validation[None, ...],
    validation_features=feature,
    validation_freqs=frequencies_validation,
    passivity_features=feature,
    passivity_freqs=passivity_frequencies,
    simulation_frequency_range=(network.f.min(), network.f.max()),
)
print(f"D shift: {float(shifts.conductance):.3e}; E shift: {float(shifts.slope):.3e}")
print(passive_report.summary())

### Four-port results

The prescribed scikit-rf fit captures the resonances more accurately and at lower order than this automatic AAA/Y configuration. Its rational passivity test still finds an out-of-band violation.

For the AAA/Y fit, shifting the asymptotic terms improves sampled passivity but doesn't fix the missing accuracy. Improving that fit would require revisiting the poles, weighting, fitting domain, or available data.

The next cell checks scikit-rf's converted Y poles. In the executed comparison, the largest real part was about +8.01×10¹¹ rad/s, so that realization contains a growing mode. It needs correction and revalidation before use as a passive component. No transient is run in this section.

You can vary the order or try scikit-rf's passivity enforcement, then compare held-out error and Y stability again.

[Back to contents](#contents)


In [ ]:
# Diagnostic use of scikit-rf's real state-space representation.
A_ref, B_ref, C_ref, D_ref, E_ref = vf_skrf._get_ABCDE()
violation_bands = vf_skrf.passivity_test()
print("scikit-rf rational passivity:", "PASS" if not len(violation_bands) else "FAIL")
print("Violation intervals (Hz):", violation_bands)
try:
    A_y_ref = A_ref - B_ref @ np.linalg.solve(np.eye(len(D_ref)) + D_ref, C_ref)
    largest_real_pole = np.linalg.eigvals(A_y_ref).real.max()
    print("Converted Y max real pole (rad/s):", largest_real_pole)
    print("Y stability:", "PASS" if largest_real_pole < 0 else "FAIL")
except np.linalg.LinAlgError as error:
    print("S-to-Y conversion: FAIL", error)
print("Passive component admission requires both checks AND acceptable validation error.")


<a id="active"></a>
## 3. Active transmitter

The transmitter in scikit-rf's [190 GHz example](https://scikit-rf.readthedocs.io/en/latest/examples/vectorfitting/vectorfitting_ex2_190ghz_active.html) has gain and different forward and reverse responses.

We'll allow both activity and nonreciprocity in this fit. Accuracy, causal behavior, and a suitable stable realization still matter, as does the frequency range of the intended simulation.

[Back to contents](#contents)


In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import skrf
from skrf.vectorFitting import VectorFitting

from circulax.fitting import (
    evaluate_sparameter_model,
    fit_with_delay,
    surface_from_fit,
    validate_surface_fit,
    y_to_s,
)

data_path = Path("examples/fitting/data/190ghz_tx_measured.s2p")
if not data_path.exists():
    data_path = Path("data/190ghz_tx_measured.s2p")
network = skrf.Network(data_path)
freqs = network.f
S = network.s
print(f"{len(freqs)} samples, {freqs[0] / 1e9:.0f}--{freqs[-1] / 1e9:.0f} GHz")

### Check gain and directionality

A passive network's S-matrix has singular values no larger than one. An amplifier can exceed that bound. We also need to fit both transmission directions: mirroring one matrix triangle would discard their difference.


In [ ]:
maximum_singular_value = np.linalg.svd(S, compute_uv=False)[..., 0].max()
maximum_nonreciprocity = np.max(np.abs(S[:, 0, 1] - S[:, 1, 0]))
best_reciprocal = 0.5 * (S + np.swapaxes(S, -1, -2))
reciprocal_error_floor = np.linalg.norm(best_reciprocal - S) / np.linalg.norm(S)

print(f"maximum singular value:       {maximum_singular_value:.3f}")
print(f"maximum |S21 - S12|:          {maximum_nonreciprocity:.3f}")
print(f"best reciprocal-model NRMSE:  {reciprocal_error_floor:.1%}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for row in range(2):
    for col in range(2):
        axes[row, col].plot(freqs / 1e9, 20 * np.log10(np.maximum(np.abs(S[:, row, col]), 1e-15)))
        axes[row, col].set_title(f"S{row + 1}{col + 1}")
        axes[row, col].set_ylabel("magnitude (dB)")
        axes[row, col].grid(True)
for axis in axes[-1]:
    axis.set_xlabel("frequency (GHz)")
fig.tight_layout()

### Compare S and Y fitting with scikit-rf

Scikit-rf can fit all ordered responses of a nonreciprocal network. Here we run its automatic fitter in S and Y. S fitting targets the measured scattering response directly; Y fitting targets the admittance relation used in Circulax's circuit equations.

Compare the reconstructed S errors below, then the Y poles at the end of the section. Stable S poles alone don't guarantee stable poles after conversion.


In [ ]:
def skrf_response(parameter_type):
    fitter = VectorFitting(network)
    started = time.perf_counter()
    fitter.auto_fit(parameter_type=parameter_type)
    elapsed = time.perf_counter() - started
    response = np.empty_like(S)
    for row in range(2):
        for col in range(2):
            response[:, row, col] = fitter.get_model_response(row, col, freqs)
    if parameter_type == "y":
        response = np.stack([y_to_s(value, 50.0) for value in response])
    return fitter, response, elapsed


vf_s, S_skrf, time_skrf_s = skrf_response("s")
vf_y, S_skrf_y, time_skrf_y = skrf_response("y")

In [ ]:
def metrics(prediction):
    error = prediction - S
    return {
        "model NRMSE": np.linalg.norm(error) / np.linalg.norm(S),
        "RMS per S entry": np.sqrt(np.mean(np.abs(error) ** 2)),
        "maximum |dS|": np.max(np.abs(error)),
    }


for label, fitter, prediction, elapsed in (
    ("scikit-rf, direct S", vf_s, S_skrf, time_skrf_s),
    ("scikit-rf, Y then S", vf_y, S_skrf_y, time_skrf_y),
):
    result = metrics(prediction)
    order = fitter.get_model_order(fitter.poles)
    print(label)
    print(f"  order={order}, elapsed={elapsed:.3f} s")
    print("  " + ", ".join(f"{key}={value:.4g}" for key, value in result.items()))

<a id="active-fit"></a>
### Fit the active device in Circulax

For this experimental fit:

- `reciprocal=False` fits all four ordered responses.
- `enforce_passive=False` leaves the gain intact.
- `delay_mode="auto"` selects `"none"` for this nonreciprocal device. The measurement doesn't justify removing the same port-delay sum from both transmission directions.

AAA finds initial stable poles from the dominant response. Contribution ranking removes weak conjugate pairs, then vector-fitting iterations refine the remaining common poles against all complex S responses. A state-space feedback transformation converts the result to Y while preserving the fitted S response.


In [ ]:
started = time.perf_counter()
ss, delay, fit_metadata = fit_with_delay(
    S,
    freqs,
    tol=1e-8,
    mmax=40,
    reciprocal=False,
    enforce_passive=False,
    delay_mode="auto",
    fit_domain="s",
    s_refinement_iterations=4,
    max_poles=20,
    pole_count_candidates=tuple(range(10, 32, 2)),
    verbose=False,
)
time_circulax = time.perf_counter() - started
S_circulax = evaluate_sparameter_model(ss, freqs, delay)

print({key: value for key, value in fit_metadata.items() if key != "pole_sweep"})
print(f"elapsed={time_circulax:.3f} s")
print(metrics(S_circulax))

sweep = fit_metadata["pole_sweep"]
print()
print("Vmapped fixed-pole screening (before pole relocation):")
for count, error in zip(sweep.retained_counts, sweep.normalized_rmse, strict=True):
    print(f"  {int(count):2d} poles: {float(error):.3%} NRMSE")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for row in range(2):
    for col in range(2):
        axis = axes[row, col]
        axis.plot(freqs / 1e9, 20 * np.log10(np.maximum(np.abs(S[:, row, col]), 1e-15)), label="measured")
        axis.plot(freqs / 1e9, 20 * np.log10(np.maximum(np.abs(S_circulax[:, row, col]), 1e-15)), "--", label="Circulax")
        axis.set_title(f"S{row + 1}{col + 1}")
        axis.set_ylabel("magnitude (dB)")
        axis.grid(True)
for axis in axes[-1]:
    axis.set_xlabel("frequency (GHz)")
axes[0, 0].legend()
fig.tight_layout()

<a id="active-validation"></a>
### Validate the active fit

The validation settings allow activity and nonreciprocity, so the checks preserve intentional gain. This fit meets the selected error threshold, but the report also notes that we haven't supplied an independent holdout sweep.

Component pole stability doesn't settle whether the amplifier will be stable with a particular source, load, or feedback connection. That requires a loaded-circuit analysis.


In [ ]:
omega_scale = 2 * np.pi * freqs[-1]
surface = surface_from_fit(ss, delay, omega_scale)
features = np.ones((1, 1))
report = validate_surface_fit(
    surface,
    S[None, ...],
    features,
    freqs,
    simulation_frequency_range=(freqs[0], freqs[-1]),
    expected_reciprocal=False,
    expected_passive=False,
)
print(report.summary())

### Transmitter results

Enforcing passivity on this transmitter would suppress the gain we're trying to model.

The final diagnostic compares the Y poles from scikit-rf's two fitting domains. In the executed comparison, the S fit converted to a Y realization with maximum pole real part about +2.71×10¹¹ rad/s. The direct Y fit had a maximum of about −4.76×10¹⁰ rad/s. Direct Y fitting gave stable component poles in this case; its accuracy still needs to be checked against the application requirements.

An unstable internal mode is different from amplifier gain, and external feedback can make a circuit oscillate even with stable component poles. We haven't run those loaded transient studies here. A separate sweep or a reserved frequency block would also help test generalization, especially before extrapolating this 140–220 GHz dataset toward DC or 500 GHz.

For a new S fit through the coefficient API, use `ModelFitOptions(reciprocal=False, enforce_passivity=False)`. The experimental configuration above uses a different fitting path and may give different results.

[Back to contents](#contents)


In [ ]:
# The S fit needs conversion; the Y fit already represents admittance.
for label, fitted, needs_conversion in [
    ("scikit-rf S fit", vf_s, True),
    ("scikit-rf Y fit", vf_y, False),
]:
    A_ref, B_ref, C_ref, D_ref, E_ref = fitted._get_ABCDE()
    try:
        if needs_conversion:
            if np.any(E_ref != 0):
                raise ValueError("This diagnostic requires a proper S model.")
            A_ref = A_ref - B_ref @ np.linalg.solve(np.eye(len(D_ref)) + D_ref, C_ref)
        largest_real_pole = np.linalg.eigvals(A_ref).real.max()
        print(label, "Y max Re(pole):", largest_real_pole, "rad/s;",
              "stable" if largest_real_pole < 0 else "UNSTABLE")
    except (ValueError, np.linalg.LinAlgError) as error:
        print(label, "conversion check failed:", error)
print("Passivity is deliberately NOT an admission requirement for this active transmitter.")


<a id="delay-aware"></a>
## 4. Fit with explicit delays

This passive two-port has a known one-pole core and a one-way delay of 0.03 s at each port. Transmission picks up 0.06 s; a reflection at either port would also pick up 0.06 s.

We'll fit the same training data with scikit-rf, first directly and then after removing those delays. The ring-slot example didn't make this comparison or specify a known propagation delay.

The saved coefficients contain both the rational core and the delays. `evaluate` returns the full terminal response; `evaluate_core` returns the core alone. `component_from_coefficients` returns a component class for ordinary zero-delay models, or a `Circuit` containing the core and transmission lines. Use `compile_circuit(models_map=...)` for either result; a returned `Circuit` can't be instantiated as `Model()` or passed directly to low-level `compile_netlist`.


In [ ]:
import tempfile
from pathlib import Path
import jax.numpy as jnp
from circulax import compile_circuit
from circulax.fitting import ModelCoefficients, ModelFitOptions, fit_model, component_from_coefficients

delay_freqs = np.linspace(0, 10, 151)
delay_truth = ModelCoefficients(
    np.array([-5.0]), np.array([[[0.0], [1.0]], [[1.0], [0.0]]]),
    np.array([[0.0, 0.2], [0.2, 0.0]]), port_delays=[0.03, 0.03],
)
delay_data = delay_truth.evaluate(delay_freqs)
delay_limits = dict(normalized_rmse=1e-4, max_absolute_error=1e-3)
delay_baseline = fit_model(delay_data, delay_freqs, options=ModelFitOptions(**delay_limits))
delay_separated = fit_model(delay_data, delay_freqs, options=ModelFitOptions(
    delay_mode="supplied", port_delays=(0.03, 0.03), **delay_limits,
))
assert len(delay_separated.poles) < len(delay_baseline.poles)
for label, artifact in [("rational only", delay_baseline), ("explicit delays", delay_separated)]:
    print(label, "core poles:", len(artifact.poles),
          "NRMSE:", artifact.metadata["training_nrmse"])

# A separate holdout grid is used only after fitting.
delay_holdout = np.linspace(0.025, 9.975, 200)
np.testing.assert_allclose(delay_separated.evaluate(delay_holdout),
                           delay_truth.evaluate(delay_holdout), atol=1e-5)
with tempfile.TemporaryDirectory() as delay_directory:
    delay_path = Path(delay_directory) / "delayed.npz"
    delay_separated.save(delay_path)
    delay_model = component_from_coefficients(delay_path)
delay_circuit = compile_circuit(
    {"instances": {"dut": {"component": "measured"}}, "connections": {},
     "ports": {"p1": "dut,p1", "p2": "dut,p2"}},
    models_map={"measured": delay_model}, g_leak=0,
)
delay_sp = delay_circuit.sp(ports=["p1", "p2"], freqs=jnp.asarray(delay_holdout))
np.testing.assert_allclose(delay_sp, delay_truth.evaluate(delay_holdout), atol=1e-5)
plt.figure(figsize=(8, 3))
plt.plot(delay_freqs, np.unwrap(np.angle(delay_data[:, 1, 0])), label="full terminal phase")
plt.plot(delay_freqs, np.unwrap(np.angle(delay_separated.evaluate_core(delay_freqs)[:, 1, 0])), label="rational core phase")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Transmission phase (radians)")
plt.title("One-way port delays: 0.03 s + 0.03 s")
plt.legend()
plt.tight_layout()


### Delay-fit results

Removing the known delays gives a lower-order fit at the same complex-S error target. To compare simulation costs too, run:

```bash
pixi run python -m benchmarks.fitting.bench_delay_separation
```

The benchmark measures fitting, conversion, compilation, warm evaluation, and transient time separately. Core states often repeat per port, and the lines add variables and history storage, so pole count alone doesn't determine runtime.

Supplied delays also work for unequal port delays and active or nonreciprocal cores. Optional enforcement acts on the core, and the final error check uses the original S data. The assembled lines work in DC, AC, HB, and transient; the older `rational_delay_component` is only a frequency-domain reference.

Automatic inference (`delay_mode="auto"`) requires an `auto_max_delay` bound on the transmission-delay sum. It currently considers passive reciprocal two-ports with negligible reflections and sufficiently linear, nonzero transmission. Candidates split the delay equally between ports and must improve on a valid baseline; ties keep the baseline. If inference is declined, the baseline is used if it passes. If neither passes, fitting raises an error. See the [API documentation](../../docs/fitting_api.md) for the checks and diagnostics.

A phase slope and sampled passivity don't prove that the de-embedded core is causal. This known-delay example also doesn't establish performance on measured networks or compare against Chinea et al.'s more general method. Exploring that formulation remains a possible next step for multiple paths and internal reflections.

For a noisy cable dataset with frequency-dependent loss, supplied propagation delay, and a fitted pulse response, see the [transmission-line example](../electrical/time_delay.ipynb).


<a id="summary"></a>
## Summary and references

The ring-slot example produces a compact fit that passes the physical checks. The four-port example shows the limits of correcting passivity when the fit itself is inaccurate. The transmitter needs settings that preserve gain and directionality. Finally, the known-delay example reduces rational order by separating propagation from the core dynamics.

For a model you plan to simulate, check held-out error, the relevant physical properties, and the actual circuit realization. Out-of-band behavior needs its own data or physical justification.

- [scikit-rf vector-fitting tutorial](https://scikit-rf.readthedocs.io/en/latest/tutorials/VectorFitting.html)
- Chinea, Triverio, and Grivet-Talocia, *Delay-Based Macromodeling of Long Interconnects From Frequency-Domain Terminal Responses*, IEEE Transactions on Advanced Packaging, 33(1), 246–256 (2010). [Paper](https://www.waves.utoronto.ca/triverio/papers/jnl-2010-tadvp-vfdelays.pdf), [DOI](https://doi.org/10.1109/TADVP.2008.2010525).
- [Fitting API](../../docs/fitting_api.md)
- [Rational-model enforcement and time-domain implications](../../docs/rational_model_enforcement.md)
- [Timing and enforcement experiments](../../benchmarks/fitting/README.md)

[Back to contents](#contents)
